In [ ]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [ ]:
# 데이터 확인하기 2025.11.21 zero_count_rate > 99% 이상인 컬럼 제거 후 RandomForest 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval

In [23]:
# 데이터 로딩
train, test = load_data()
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 

In [24]:
# Data 전처리 1. zero_count_rate이 95%인 컬럼 제거하기 
X_features = remove_zero_columns(X_features, 0.95)


Zero Value Analysis (Threshold: 95.0% = 72,219 rows)
Total rows: 76,020
Total columns: 369

                                     Summary 정보 (zero_count 내림차순)                                     
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020         100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020         100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020         100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020         100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020         100.00%
              saldo_var2_ult1       0        1      0.000000     76020      100.00%       76020         100.00%
                   i

In [25]:
# TEST Data 전처리 1. zero_count_rate이 95%인 컬럼 제거하기 
X_test = remove_zero_columns(X_test, 0.95)


Zero Value Analysis (Threshold: 95.0% = 72,027 rows)
Total rows: 75,818
Total columns: 369

                                     Summary 정보 (zero_count 내림차순)                                     
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate
              saldo_var2_ult1       0        1      0.000000     75818      100.00%       75818         100.00%
saldo_medio_var13_medio_hace3       0        1      0.000000     75818      100.00%       75818         100.00%
                   ind_var2_0       0        1      0.000000     75818      100.00%       75818         100.00%
                     ind_var2       0        1      0.000000     75818      100.00%       75818         100.00%
        num_reemb_var17_hace3       0        1      0.000000     75818      100.00%       75818         100.00%
        num_reemb_var33_hace3       0        1      0.000000     75818      100.00%       75818         100.00%
         num_reemb_v

In [26]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [27]:
# 스케일링
X_train_scaled, X_test_scaled, scaler = scale_data(X_features, X_test)


In [ ]:
# # 레이블의 분포 확인
# cust_cnt = y_labels.value_counts()
# print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# # 불만족고객의 비율
# cust_rate = cust_cnt[1] / cust_cnt.sum()
# print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [ ]:
# first testing model
# XGBoost (xgb) : yjh, kjh
# LightGBM(lgbm) : lsj, ujm
# Random Forest(rf) : lkj, kjh
# Logistic Regression(lr) : yjh, ujm


In [28]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features, 
  y_labels,
)


In [29]:
# Model 학습, 평가
from sklearn.ensemble import RandomForestClassifier 

model_name = 'RandomForest_95per_basic'

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 100,
  max_depth    = 8, # RF : 애가 핵심이야 Tree 계열이니까~ 약한 Tree로 만들어야해 그래서 max_depth로 자른거
  n_jobs       = -1 # 병렬처리 여부 
)


# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)

✓ 모델 저장 완료: models\RandomForest_95per_basic.pkl
  파일 크기: 1.74 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8231, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0017, F1: 0.0033
오차행렬:
[[14601     1]
 [  601     1]]


In [30]:
model_name = 'RandomForest_95per_HP_maxDepth10'

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 100,
  max_depth    = 10, # RF : 애가 핵심이야 Tree 계열이니까~ 약한 Tree로 만들어야해 그래서 max_depth로 자른거
  n_jobs       = -1 # 병렬처리 여부 
)


# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)

✓ 모델 저장 완료: models\RandomForest_95per_HP_maxDepth10.pkl
  파일 크기: 3.40 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8317, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0017, F1: 0.0033
오차행렬:
[[14601     1]
 [  601     1]]


In [31]:
model_name = 'RandomForest_95per_HP_maxDepth10_classWeight'

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 100,
  max_depth    = 10, 
  class_weight = {0:1, 1:2}, # 클래스별 가중치
  n_jobs       = -1 # 병렬처리 여부 
)


# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)

✓ 모델 저장 완료: models\RandomForest_95per_HP_maxDepth10_classWeight.pkl
  파일 크기: 3.81 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8364, 정확도: 0.9605, 정밀도: 1.0000, 재현율: 0.0033, F1: 0.0066
오차행렬:
[[14602     0]
 [  600     2]]


In [32]:
model_name = 'RandomForest_95per_HP_ne300_maxDepth20_classWeight'

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 300,
  max_depth    = 20, 
  class_weight = {0:1, 1:2}, # 클래스별 가중치
  n_jobs       = -1 # 병렬처리 여부 
)


# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)

✓ 모델 저장 완료: models\RandomForest_95per_HP_ne300_maxDepth20_classWeight.pkl
  파일 크기: 60.00 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8396, 정확도: 0.9605, 정밀도: 0.5714, 재현율: 0.0066, F1: 0.0131
오차행렬:
[[14599     3]
 [  598     4]]
